# 한 줄짜리 Research Agent 구축하기

조사 작업은 전문가의 시간을 몇 시간씩 소모합니다. 시장 분석가는 경쟁 정보를 수동으로 수집하고, 법무팀은 규제 변화를 추적하며, engineer는 문서 전반에서 bug report를 조사합니다. 핵심 과제는 정보를 찾는 것 자체가 아니라 방금 발견한 내용을 바탕으로 다음에 무엇을 검색할지 아는 것입니다.

Claude Agent SDK를 사용하면 미리 정의한 workflow 없이 외부 system을 자율적으로 탐색하는 agent를 구축할 수 있습니다. 고정된 단계를 따르는 전통적인 workflow 자동화와 달리, research agent는 발견한 내용에 따라 전략을 조정합니다. 유망한 단서를 따라가고, 서로 상충하는 출처를 종합하며, 질문에 답할 만큼 충분한 정보를 확보했는지 판단합니다.

## 이 cookbook을 마치면 다음을 할 수 있습니다:

- 몇 줄의 코드로 정보를 자율적으로 검색하고 종합하는 research agent 구축하기

이 기반은 경쟁 분석, 기술 문제 해결, 투자 조사, 문헌 검토처럼 필요한 정보가 처음부터 제공되지 않는 모든 작업에 적용됩니다.

# Research Agent를 사용하는 이유

조사가 이상적인 agentic 사용 사례인 이유는 두 가지입니다:

1. **정보가 자체적으로 완결되어 있지 않습니다**. 입력 질문만으로는 답을 얻을 수 없습니다. Agent는 필요한 정보를 수집하기 위해 외부 system(검색 engine, database, API)과 상호작용해야 합니다.
2. **탐색하면서 경로가 드러납니다**. Workflow를 미리 정할 수 없습니다. Agent가 기업 재무 정보와 규제 제출 문서 중 무엇을 검색해야 하는지는 business model에 대해 발견한 내용에 달려 있습니다. 최적 전략은 조사하면서 드러납니다.

가장 단순한 형태의 research agent는 web을 검색하고 발견 사항을 종합합니다. 아래에서는 Claude Agent SDK의 내장 web search tool을 사용하여 바로 그런 agent를 몇 줄의 코드로 구축합니다.

참고: [Claude Code의 내장 tool](https://docs.claude.com/en/docs/claude-code/settings#tools-available-to-claude) 전체 목록도 볼 수 있습니다.

# 사전 요구사항

이 가이드를 따라 하기 전에 다음을 준비하세요:

**필수 지식**

* Python 기초 - async/await, function, 기본 data structure에 익숙함
* Agentic pattern에 대한 기본 이해 - agent가 처음이라면 먼저 [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)를 읽는 것을 권장합니다

**필수 Tool**

* Python 3.11 이상
* Anthropic API key [(여기에서 발급)](https://console.anthropic.com)

**권장 사항:**
* Claude Agent SDK 개념에 익숙함
* LLM의 tool use pattern 이해


## 설정

먼저 필요한 dependency를 설치합니다:

In [ ]:
%%capture
%pip install -U claude-agent-sdk python-dotenv

참고: .env 파일에 다음이 포함되어 있는지 확인하세요:

```bash
ANTHROPIC_API_KEY=your_key_here
```

환경 변수를 load하고 client를 구성합니다:

In [2]:
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-opus-4-6"

## 첫 Research Agent 구축하기

가능한 한 단순한 구현부터 시작하겠습니다. Web을 검색하고 발견 사항을 종합할 수 있는 research agent입니다. Claude Agent SDK를 사용하면 몇 줄의 코드만 필요합니다.

핵심은 stateless agent 상호작용을 생성하는 query() function입니다. Claude에 WebSearch라는 tool 하나를 제공하고, 조사 질문에 따라 언제 어떻게 사용할지 자율적으로 결정하게 합니다.

In [3]:
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
)

from claude_agent_sdk import ClaudeAgentOptions, query

messages = []
async for msg in query(
    prompt="Research the latest trends in AI agents and give me a brief summary and relevant citiations links.",
    options=ClaudeAgentOptions(model=MODEL, allowed_tools=["WebSearch"]),
):
    print_activity(msg)
    messages.append(msg)

🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


In [4]:
display_agent_response(messages)

## 여기서 일어나는 일:

- `query()`는 single-turn agent 상호작용을 생성합니다(대화 memory 없음)
- `allowed_tools=["WebSearch"]`는 승인 요청 없이 web을 검색할 권한을 Claude에 제공합니다
- Agent는 언제 검색할지, 어떤 query를 실행할지, 결과를 어떻게 종합할지 자율적으로 결정합니다

**`utils.agent_visualizer`의 시각화 utility:**
- `print_activity()` - agent의 행동(tool 호출, thinking)을 실시간으로 보여 줍니다
- `display_agent_response()` - 최종 응답을 style이 적용된 HTML card로 rendering합니다
- `visualize_conversation()` - 전체 대화의 timeline view를 생성합니다

이것으로 끝입니다! 단 몇 줄의 코드로 작동하는 research agent를 만들었습니다. Agent는 관련 정보를 검색하고, 유망한 단서를 추가 조사하며, citation과 함께 종합한 요약을 제공합니다.

query() function은 stateless agent 상호작용을 생성합니다. 각 호출은 독립적이며 대화 memory도 이전 query의 context도 없습니다. 따라서 state를 유지하지 않고 빠른 답변이 필요한 일회성 조사 작업에 아주 적합합니다.

**Tool permission의 작동 방식:**

`allowed_tools=["WebSearch"]` parameter는 승인 요청 없이 검색할 권한을 Claude에 제공합니다. 이는 자율적인 작동에 필수적입니다:

- `Allowed tools` - Claude가 자유롭게 사용할 수 있습니다(여기서는 WebSearch)
- `Other tools` - 사용 가능하지만 사용 전에 승인이 필요합니다
- `Read-only tools` - Read 같은 tool은 기본적으로 항상 허용됩니다
- `Disallowed tools` - disallowed_tools에 tool을 추가하면 Claude의 context에서 완전히 제거됩니다

**Stateless query를 사용할 때:**

- Context가 중요하지 않은 일회성 조사 질문
- 독립적인 조사 작업의 병렬 처리
- Query마다 새로운 context를 원하는 경우

**Stateless query를 사용하지 말아야 할 때:**

- 이전 발견 사항을 바탕으로 진행하는 multi-turn 조사
- 초기 결과를 바탕으로 조사를 반복적으로 다듬기
- 지속적인 context가 필요한 복잡한 분석

visualize_conversation helper로 agent가 실제로 무엇을 했는지 살펴보겠습니다:

In [5]:
from utils.agent_visualizer import visualize_conversation

visualize_conversation(messages)

## Prototype에서 Production으로: 세 가지 핵심 개선

한 줄짜리 research agent는 작동하지만 한계가 있습니다. Memory가 없는 단일 query로는 반복적인 조사("X를 찾고, 발견한 내용을 바탕으로 Y를 분석해")를 처리할 수 없습니다. 구현을 더 개선할 세 가지 방법을 살펴보겠습니다.

**1. ClaudeSDKClient를 사용한 대화 Memory**: Stateless query는 이전 발견 사항을 바탕으로 이어갈 수 없습니다. "주요 AI startup은 어디인가요?"를 물은 뒤 "이들은 자금을 어떻게 조달하나요?"라고 물으면, 두 번째 query는 어느 startup을 뜻하는지에 대한 context가 없습니다. `ClaudeSDKClient`로 여러 query에 걸쳐 대화 기록을 유지할 수 있습니다.


**2. 전문화된 동작을 위한 System Prompt**: 조사 분야에는 특정 요구사항이 있는 경우가 많습니다. 재무 분석은 기술 뉴스 요약과 다른 수준의 엄밀함이 필요합니다. System prompt에 조사 기준, 선호하는 출처, citation 형식 또는 출력 구조를 지정하세요. 조사에 특화된 예제는 [agent prompting guide](https://github.com/anthropics/anthropic-cookbook/tree/main/patterns/agents/prompts)를 참고하세요.

**3. Read Tool을 사용한 Multimodal 조사**: 실제 조사는 text만 다루지 않습니다. 시장 보고서에는 chart가, 기술 문서에는 diagram이 있으며, 경쟁 분석에는 screenshot 비교가 필요합니다. Claude가 image, PDF 및 다른 시각 콘텐츠를 분석할 수 있도록 `Read` tool을 활성화하세요.

Research agent에 이 세 가지 변경을 구현하겠습니다.

In [6]:
from claude_agent_sdk import ClaudeSDKClient

# 조사 품질을 위한 citation 요구사항이 있는 system prompt
RESEARCH_SYSTEM_PROMPT = """You are a research agent specialized in AI.

When providing research findings:
- Always include source URLs as citations
- Format citations as markdown links: [Source Title](URL)
- Group sources in a "Sources:" section at the end of your response"""

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        cwd="research_agent",
        system_prompt=RESEARCH_SYSTEM_PROMPT,
        allowed_tools=["WebSearch", "Read"],
        max_buffer_size=10 * 1024 * 1024,  # Image 처리를 위해 10MB로 증가
    )
) as research_agent:
    # 첫 query: chart image 분석
    await research_agent.query("Analyze the chart in research_agent/projects_claude.png")
    async for msg in research_agent.receive_response():
        print_activity(msg)
        messages.append(msg)

    # 두 번째 query: web 검색으로 chart 분석 결과 검증/맥락 보충
    await research_agent.query(
        "Based on the chart analysis, search for recent news or data that validates or provides context for these findings. Include source URLs."
    )
    async for msg in research_agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: Glob()
✓ Tool completed
🤖 Thinking...
🤖 Using: Read()
✓ Tool completed
🤖 Thinking...
🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
🤖 Using: WebSearch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


### 🔧 큰 응답 및 Buffer 한도 처리하기

Image나 큰 data를 다룰 때 buffer overflow 오류가 발생할 수 있습니다:

```
Fatal error in message reader: Failed to decode JSON: JSON message exceeded maximum buffer size of 1048576 bytes
```

**발생 이유:**
- 기본 `max_buffer_size`는 1MB(1,048,576 bytes)입니다
- Image는 message에서 base64로 encoding되어 크기가 크게 늘어납니다
- Chart image(disk에서 약 200KB)는 base64 encoding 시 약 270KB 이상이 되며, 여기에 message overhead가 더해집니다

**해결 방법:**
Image나 큰 tool 출력을 다룰 때 `ClaudeAgentOptions`의 `max_buffer_size`를 더 큰 값(예: 10MB)으로 설정하세요.

**모범 사례:**
- 사용 사례에 맞춰 buffer 크기를 설정하세요. 일반적인 multimodal 작업은 10MB, 대규모 문서 처리는 더 큰 값을 사용합니다
- 전체 image를 전달해야 하는지 검토하세요. 설명이나 작은 thumbnail이면 충분한 경우도 있습니다
- Buffer 오류를 모니터링하고 그에 맞게 조정하세요
- 검증 가능한 조사 출력을 보장하도록 system prompt에 citation 요구사항을 포함하세요

## 여기서 일어나는 일:

이 예제는 대화 memory, citation을 고려한 system prompt, multimodal 분석이라는 세 가지 개선을 모두 결합합니다.

**핵심 구성 요소:**

| 구성 요소 | 목적 |
|-----------|---------|
| `ClaudeSDKClient` | 여러 query에 걸쳐 대화 state 유지 |
| `RESEARCH_SYSTEM_PROMPT` | Citation 형식 및 출처 URL을 요구 |
| `allowed_tools=["WebSearch", "Read"]` | Web 검색 및 image/document 분석 활성화 |
| `max_buffer_size=10MB` | Overflow 없이 base64-encoded image 처리 |

**실행 흐름:**

1. **첫 query** - `Read` tool로 chart image를 분석합니다
2. **첫 response loop** - agent가 완료할 때까지 모든 message를 수집합니다
3. **두 번째 query** - web을 검색하여 chart 분석 결과를 검증하고 맥락을 보충합니다
4. **Context 상속** - 두 번째 query가 첫 query의 chart 분석을 기억합니다

**`query()` 대신 `ClaudeSDKClient`를 사용하는 이유:**

`async with ClaudeSDKClient()` context manager는 대화 state를 유지합니다. 각 `receive_response()` 호출은 이전 context를 바탕으로 진행합니다. 이는 독립적인 stateless session을 생성하는 `query()`와 다릅니다.

In [7]:
visualize_conversation(messages)

Pattern,Implication
Claude Code dominates Startup Work,Developers building products at startups prefer the code-focused interface for rapid development
Claude.ai leads in educational contexts,"The conversational nature of Claude.ai makes it more approachable for learning, research, and coursework"
Personal Projects are universal,Both platforms serve individual developers working on side projects equally well
Enterprise usage is balanced,Both products have found their place in professional enterprise environments
Metric,2025 Data
AI-generated/assisted code,41% of all code globally
Developers using AI coding assistants,82% daily or weekly
Market size projection,$30.1 billion by 2032
Google's AI-assisted code,25%
Chart Finding,Validation Status


## Production을 위한 구축

Jupyter Notebook은 학습에 좋지만 production system에는 재사용 가능한 module이 필요합니다. Research agent를 명확한 interface를 갖춘 `research_agent/agent.py`로 package화했습니다:

### 핵심 function:

- `print_activity()` - agent가 하는 일을 실시간으로 보여 줍니다(공유 utility에서 import)
- `get_activity_text()` - logging이나 monitoring 같은 custom handler용 activity text를 추출합니다
- `send_query()` - 내장 activity 표시가 있는 research query의 주 진입점

### 내장된 모범 사례:

Module에는 다음을 보장하는 `RESEARCH_SYSTEM_PROMPT`가 포함됩니다:
- 출처 URL을 항상 citation으로 포함
- 깔끔한 rendering을 위해 citation을 Markdown link로 formatting
- "Sources:" section에 모든 reference를 모음

### 표시 제어:

`send_query()` function에는 `display_result` parameter가 있습니다(기본값: `True`):
- `display_result=True` - Jupyter Notebook에 style이 적용된 HTML card를 rendering합니다
- `display_result=False` - Programmatic 사용을 위해 text 결과만 반환합니다

이제 어떤 Python script에서도 이 agent를 사용할 수 있습니다!

대화 context가 중요하지 않은 독립적인 질문에 사용합니다.

Module은 다음을 자동으로 처리합니다:
- 실행 중 activity 표시
- 새 대화를 위한 context 초기화
- 최종 응답의 style이 적용된 HTML rendering

In [8]:
from research_agent.agent import send_query

# Module 내부에서 activity 표시, context 초기화, 결과 시각화를 처리함
result = await send_query("What is the Claude Code SDK? Only do one websearch and be concise")

🤖 Using: WebSearch()
✓ Tool completed
🤖 Thinking...


이제 같은 대화를 재사용하는 multi-turn 대화를 시험합니다.

Multi-turn 대화는 원활하게 작동합니다. `continue_conversation=True`만 전달하세요:

In [9]:
result1 = await send_query("What is Anthropic? Only do one websearch and be concise")

🤖 Using: WebSearch()
✓ Tool completed
🤖 Thinking...


In [10]:
# continue_conversation=True를 설정하여 대화를 이어가고 더 깊이 조사
result2 = await send_query(
    "What are some of their products?",
    continue_conversation=True,
)

🤖 Thinking...


## 결론

### 구축한 것

이 cookbook에서는 점차 정교해지는 세 가지 research agent를 구축했습니다:

- Stateless research agent - 독립적인 조사 작업을 위한 한 줄 query
- Memory가 있는 stateful agent - 이전 발견 사항을 바탕으로 진행하는 multi-turn 조사
- Production module - Application 통합을 위한 재사용 가능한 research function

### 핵심 사항

**Stateless query(query())를 사용할 때:**

- 독립적인 조사 질문
- 서로 관련 없는 작업의 병렬 처리
- 매번 새로운 context가 필요한 경우

**Stateful agent(ClaudeSDKClient)를 사용할 때:**

- 이전 발견 사항을 바탕으로 진행하는 multi-turn 조사
- 조사를 반복적으로 다듬기
- 지속적인 context가 필요한 복잡한 분석

Research agent는 정보가 자체적으로 완결되지 않고 탐색 중에 최적 workflow가 드러나는 상황에서 탁월합니다. 경쟁 분석, 기술 문제 해결, 문헌 검토, 탐사 보도가 모두 이 pattern에 해당합니다.

### 다음 단계

자율적인 조사의 기반은 enterprise 수준 multi-agent system을 준비하는 데 도움이 됩니다. 다음 Notebook에서는 다음을 배우게 됩니다:

조정 agent 아래에서 전문 subagent orchestration하기
Hook 및 custom command로 governance 구현하기
서로 다른 이해관계자(경영진과 기술팀)에 맞게 output style 조정하기

다음: [01_The_chief_of_staff_agent.ipynb](01_The_chief_of_staff_agent.ipynb) - 단일 agent에서 multi-agent orchestration으로.